# Lesson 08 · Mixture models and expectation-maximization

**Source:** *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 9 (pp. 75–88)

**Question:** Suppose we didn't know who the pitchers were. Could batting records alone separate pitchers from everyone else, and how should we shrink a player when we don't know which group he belongs to?

**Goal:** Fit a two-component beta-binomial mixture with an EM algorithm written from scratch, assign players to clusters with posterior probabilities, and shrink with a mixture prior.

**Contents**

1. Setup: pitchers back in
2. One beta doesn't fit
3. A beta-binomial fitting function
4. EM by hand: maximize, expect, repeat
5. Posterior cluster probabilities
6. How well did it find the pitchers?
7. Shrinkage with a mixture
8. Exercises

**Tags:** `mixture-models` `expectation-maximization` `clustering` `beta-binomial` `posterior-probability` `shrinkage`

**Before you start:** Lessons 02 and 06.

*How this notebook works:* each **Your turn** prompt is followed by an empty cell for your code. **Check** boxes give values to compare against; they come from the book, and your numbers can differ slightly because the Lahman data now runs through more recent seasons. **Reflect** prompts are for short written answers. Every prompt is numbered *lesson.section.question* (for example **2.3.1**), so you can refer to it; empty code cells and answer cells carry the same number.

In [ ]:
# --- Setup: run this cell first ------------------------------------------
# Windows + conda: put the environment's DLL folders on PATH, so plotting
# can't crash the kernel when Jupyter wasn't started from the activated env.
import os, sys
for sub in ["Library\\mingw-w64\\bin", "Library\\usr\\bin", "Library\\bin", "Scripts", "bin"]:
    p = os.path.join(sys.prefix, sub)
    if os.path.isdir(p) and p not in os.environ["PATH"]:
        os.environ["PATH"] = p + os.pathsep + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import stats, optimize, special

import eb_data   # helpers in this folder: loading the Lahman data, saving results

rng = np.random.default_rng(2017)
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (7, 4)
print(f"numpy {np.__version__} | pandas {pd.__version__} | scipy {scipy.__version__}")

# National League players since 1980, pitchers included (flagged in is_pitcher)
career = eb_data.career_with_pitchers(league="NL", since=1980)
print(f"{len(career):,} players, {career['is_pitcher'].mean():.0%} pitchers")

## 1. Setup

We'll fit only on players with **at least 20 at-bats**. Keep `is_pitcher` aside: the algorithm must not see it. We use it only to grade the result at the end.

**8.1.1 · Your turn.** Make `fit_data = career[career.AB >= 20]`.

In [ ]:
# 8.1.1 · Your code here

## 2. One beta doesn't fit

**8.2.1 · Your turn.** Plot a histogram of raw averages. Overlay the density of a single beta fitted to these players, then the same histogram split by `is_pitcher`.

In [ ]:
# 8.2.1 · Your code here

## 3. A beta-binomial fitting function

**8.3.1 · Your turn.** Write `fit_bb_mle(H, AB, weights=None)` returning `(alpha, beta)` by maximum likelihood. You've done this in Lesson 02; now wrap it as a reusable function. (`weights` are for the exercise; let them default to 1.)

In [ ]:
# 8.3.1 · Your code here

> **Check:** on all players with AB ≥ 20: α ≈ 12.8, β ≈ 45.5.

## 4. EM by hand: maximize, expect, repeat

We don't know the clusters *or* their parameters. EM alternates:

- **M-step (maximize):** given the cluster assignments, fit a beta-binomial to each cluster.
- **E-step (expect):** given the fitted clusters, compute each player's likelihood under each one (`stats.betabinom.pmf`) and assign him to the more likely cluster.

The book uses *hard* assignments, the most likely cluster. Start from random assignments.

**8.4.1 · Your turn.** Randomly assign each player to cluster A or B (seed your generator), then run one M-step. Plot the two fitted densities over the histogram.

In [ ]:
# 8.4.1 · Your code here

**8.4.2 · Your turn.** Run one E-step. Look at Jeff Abbott (11/42) and Kyle Abbott (3/31): which cluster does each go to?

In [ ]:
# 8.4.2 · Your code here

**8.4.3 · Your turn.** Wrap both steps in a loop for 10 iterations. Store the parameters and cluster sizes at every iteration, and print progress as it runs (iteration, α and β for each cluster, players that changed cluster).

In [ ]:
# 8.4.3 · Your code here

**8.4.4 · Your turn.** Plot the histograms of the assignments at iterations 0, 1, 2, 5 and the last, plus each cluster's α, β and mean across iterations.

In [ ]:
# 8.4.4 · Your code here

> **Check:** the clusters settle near Beta(27, 170) (mean ≈ 0.14, the pitchers) and Beta(145, 419) (mean ≈ 0.26). Which label lands on which group depends on your random start.

## 5. Posterior cluster probabilities

By Bayes' theorem (with equal prior weight on each cluster for now):

$$\Pr(A \mid H, AB) = \frac{L_A}{L_A + L_B}.$$

**8.5.1 · Your turn.** For the players with exactly AB = 100 (the book has six), plot each one's likelihood under the two clusters as a function of H, and compute Pr(pitcher cluster).

In [ ]:
# 8.5.1 · Your code here

**8.5.2 · Your turn.** Compute the posterior cluster probability for every player with AB > 20.

In [ ]:
# 8.5.2 · Your code here

## 6. How well did it find the pitchers?

**8.6.1 · Your turn.** Keep players whose top cluster probability is at least 0.8, and cross-tabulate assigned cluster against `is_pitcher` (`pd.crosstab`).

In [ ]:
# 8.6.1 · Your code here

> **Check:** the book got about 1,697 non-pitchers and 697 pitchers right, with 106 + 55 wrong.

## 7. Shrinkage with a mixture

Each player's posterior is now a *mixture* of two betas, weighted by his cluster probabilities. The posterior mean is still easy, by linearity:

$$\hat p_i = \sum_k \Pr(k \mid \text{data}_i)\, \frac{H_i + \alpha_k}{AB_i + \alpha_k + \beta_k}.$$

**8.7.1 · Your turn.** Compute the mixture-shrunken estimate for every player. Plot it against AB (log x), next to a single-prior estimate.

In [ ]:
# 8.7.1 · Your code here

**8.7.2 · Reflect.** Players with intermediate averages barely move. Why? And why is a 95% credible interval awkward for someone whose posterior has two bumps?

*8.7.2 · Your answer:*

## 8. Exercises

**8.8.1 · Your turn.** **Soft EM.** Use the posterior probabilities as weights in the M-step (a weighted log-likelihood), and also estimate the mixing proportions π_A and π_B, using them in the E-step. Does the answer change? Does it converge faster or slower?

In [ ]:
# 8.8.1 · Your code here

**8.8.2 · Your turn.** Try three clusters. What does the third one capture, if anything?

In [ ]:
# 8.8.2 · Your code here

**8.8.3 · Reflect.** **Card connection.** Imagine a dataset of trading-card sales. Most cards are *bulk*: common cards that sell for well under a dollar. A few are *chase* cards, the rare ones collectors hunt for, selling for tens or hundreds of dollars. Separately, a card can be sold *raw* (as it came out of the pack) or *graded* (authenticated, scored for condition from 1 to 10 and sealed in a plastic case by a grading company), which usually raises its price. Where might a mixture show up in that data: bulk cards vs chase cards, raw vs graded listings? What would the clusters mean, and what would you use them for?

*8.8.3 · Your answer:*

## References

- *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 9.
- Bishop, *Pattern Recognition and Machine Learning* (2006), Chapter 9: mixture models and EM.
- McLachlan & Peel, *Finite Mixture Models* (2000).